# RAV-Med: Step 6g, SciFact judge scores per item, and the paired comparison with ERS
**Why.** Step 6a reported SciFact judge results only as aggregates, and its cached judgements could not be matched to the rebuilt items (Step 6f). This notebook judges the 1,909 SciFact items again with exactly the Step 6a prompts and model (Qwen2.5-7B-Instruct, 4-bit, greedy single forward pass), **saves the score of every item**, and runs the paired cluster bootstrap over claims (ERS minus each judge score).

**Needs a T4 GPU.** About 7,600 prompts (4 per item); expect roughly one to two hours. **Upload when asked:** `s6d_scifact_scores_full_notext.csv` (or the 6d zip). Progress is saved to a file and downloaded every `AUTO_DOWNLOAD_EVERY_MIN` minutes; if the session dies, set `RESUME_FROM_UPLOAD = True` and upload the latest `s6g_partial_cache.json`.

**Fixed before running.** (1) Prompts and settings are copied unchanged from Step 6a. (2) Reproduction check: the AUROCs recomputed here are compared with the Step 6a table (ERS 0.828; reliability prompt 0.877 for support vs contradict and 0.945 pooled; relevance prompt 0.941 pooled). Small differences are expected (4-bit numerics, batch composition); a difference above 0.01 is **reported loudly, not hidden**, and the results then need interpretation before use. Nothing is tuned.

In [ ]:
!pip install -q transformers accelerate bitsandbytes scikit-learn pandas numpy

In [ ]:
import os, re, json, time, shutil, hashlib, tarfile, zipfile, urllib.request
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
DRY_RUN = False
SEED = 42; LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"; MAX_WORDS = 250; JBATCH = 8; LOAD_4BIT = True
MAX_RUN_HOURS = 10.5; AUTO_DOWNLOAD_EVERY_MIN = 30; RESUME_FROM_UPLOAD = False
SF_DIR = os.environ.get("SF_DIR", "scifact_data"); OUT = "step6g_outputs"; os.makedirs(OUT, exist_ok=True)
SCORES6D = "s6d_scifact_scores_full_notext.csv"; PARTIAL = "s6g_partial_cache.json"; START = time.time()
def need(name):
    """The file itself, or the zip that contains it."""
    if os.path.exists(name):
        head = open(name, "rb").read(200)
        if not (name.endswith(".csv") and (head[:2] == b"PK" or b"\\x00" in head)): return
        os.remove(name); print(name, "was not a text CSV (probably a zip saved under this name); removed. Upload it again.")
    from google.colab import files; print("Upload", name, "(or the zip that contains it)"); up = files.upload()
    for n in list(up):
        if n.lower().endswith(".zip"):
            with zipfile.ZipFile(n) as z:
                for m in z.namelist():
                    if os.path.basename(m) == name:
                        with z.open(m) as src, open(name, "wb") as dst: shutil.copyfileobj(src, dst)
        elif n != name: print("ignored (not the expected file):", n)
    if not os.path.exists(name): raise FileNotFoundError(f"{name} not found in what you uploaded.")
need(SCORES6D); S6D = pd.read_csv(SCORES6D); print("6d items:", len(S6D))
# --- SciFact items: identical to Steps 6a/6d ---
if not os.path.exists(f"{SF_DIR}/corpus.jsonl"):
    urllib.request.urlretrieve("https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", "scifact.tar.gz")
    with tarfile.open("scifact.tar.gz") as t: t.extractall(".")
    if os.path.isdir("data") and not os.path.isdir(SF_DIR): os.rename("data", SF_DIR)
rd = lambda f: [json.loads(l) for l in open(f"{SF_DIR}/{f}", encoding="utf-8")]
corpus = {int(d["doc_id"]): (d["title"] + ". " + " ".join(d["abstract"])).strip() for d in rd("corpus.jsonl")}
claims_sf = rd("claims_train.jsonl") + rd("claims_dev.jsonl")
rows, mixed = [], 0
for c in claims_sf:
    ev = c.get("evidence", {})
    for d, lst in ev.items():
        labs = {e["label"] for e in lst}
        if len(labs) == 1: rows.append((c["id"], c["claim"], int(d), "SUPPORT" if "SUPPORT" in labs else "CONTRADICT"))
        else: mixed += 1
    if not ev:
        for d in c.get("cited_doc_ids", []): rows.append((c["id"], c["claim"], int(d), "NEI_ONTOPIC"))
sf = pd.DataFrame(rows, columns=["claim_id", "claim", "doc_id", "cond"])
allids = np.array(sorted(corpus)); r2 = np.random.default_rng(SEED); irr = []
cited = {c["id"]: set(map(int, c.get("cited_doc_ids", []))) | set(map(int, c.get("evidence", {}).keys())) for c in claims_sf}
for cid, g in sf[sf.cond.isin(["SUPPORT", "CONTRADICT"])].groupby("claim_id"):
    while True:
        d = int(r2.choice(allids))
        if d not in cited[cid]: break
    irr.append((cid, g.claim.iloc[0], d, "IRRELEVANT"))
sf = pd.concat([sf, pd.DataFrame(irr, columns=sf.columns)], ignore_index=True).drop_duplicates(["claim_id", "doc_id"]).reset_index(drop=True)
sf = sf[sf.doc_id.isin(corpus)].copy(); sf["passage"] = sf.doc_id.map(corpus)
k6 = set(zip(S6D.claim_id, S6D.doc_id, S6D.cond)); k_sf = set(zip(sf.claim_id, sf.doc_id, sf.cond))
print("items:", len(sf), "| in common with 6d:", len(k6 & k_sf), "| by condition:", sf.cond.value_counts().to_dict())
if not DRY_RUN: assert len(sf) == 1909 and k6 == k_sf, "The rebuilt items do not match the 6d items. Stop and send the output to the author."

In [ ]:
# --- judge model and prompts: copied unchanged from Step 6a ---
import torch
SYS = "You are a careful medical evidence reviewer. Answer with a single word: Yes or No."
QUESTIONS = {
 "rel":   "Is the passage relevant to the claim, meaning it addresses the same medical question?",
 "con":   "Does the passage contradict the claim, meaning it says something that cannot be true if the claim is true?",
 "sup":   "Does the passage support the claim?",
 "trust": "Would a careful clinician treat this passage as reliable evidence for the claim, that is, relevant to it and consistent with it?"}
def trunc(t, n=MAX_WORDS):
    w = t.split(); return t if len(w) <= n else " ".join(w[:n]) + " ..."
def _key(s, u): return hashlib.md5((LLM_NAME + "\x00" + s + "\x00" + u).encode()).hexdigest()
def _save(c):
    json.dump(c, open(PARTIAL + ".tmp", "w")); os.replace(PARTIAL + ".tmp", PARTIAL)
cache = {}
if RESUME_FROM_UPLOAD and not os.path.exists(PARTIAL):
    from google.colab import files; print("Upload your s6g_partial_cache.json"); up = files.upload()
    for n_ in up: shutil.move(n_, PARTIAL)
if os.path.exists(PARTIAL): cache = json.load(open(PARTIAL))
print("judgements already saved:", len(cache))
if DRY_RUN:
    def _h(s): return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16) / 0xffffffff
    def _score_chunk(chunk): return [_h(u) for s, u in chunk]
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    DEVICE = "cuda"; assert torch.cuda.is_available(), "Switch the runtime to a T4 GPU."
    tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, device_map={"": 0}, quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")).eval()
    YES, NO = tok.encode("Yes", add_special_tokens=False)[0], tok.encode("No", add_special_tokens=False)[0]
    def _chat(s, u):
        try: return tok.apply_chat_template([{"role": "system", "content": s}, {"role": "user", "content": u}], tokenize=False, add_generation_prompt=True)
        except Exception: return tok.apply_chat_template([{"role": "user", "content": s + "\n\n" + u}], tokenize=False, add_generation_prompt=True)
    @torch.no_grad()
    def _score_chunk(chunk):
        try:
            enc = tok([_chat(s, u) for s, u in chunk], return_tensors="pt", padding=True).to(DEVICE)
            lg = llm(**enc).logits[:, -1, :].float()[:, [YES, NO]]
            return torch.softmax(lg, -1)[:, 0].cpu().tolist()
        except torch.cuda.OutOfMemoryError:
            import gc; gc.collect(); torch.cuda.empty_cache()
            if len(chunk) == 1: raise
            h = len(chunk) // 2; return _score_chunk(chunk[:h]) + _score_chunk(chunk[h:])
PROMPTS = {k: [(SYS, f"Claim: {c}\n\nPassage: {trunc(p)}\n\n{q} Answer Yes or No.") for c, p in zip(sf.claim, sf.passage)] for k, q in QUESTIONS.items()}
allp = [p for v in PROMPTS.values() for p in v]
todo = sorted({p for p in allp if _key(*p) not in cache}, key=lambda p: len(p[1]))
print(f"{len(allp)} prompts, {len(todo)} to judge; judging starts now (a progress line prints every 10 batches)", flush=True)
t0 = time.time(); last_dl = time.time(); stopped = False
for b, i in enumerate(range(0, len(todo), JBATCH)):
    if (time.time() - START) / 3600 > MAX_RUN_HOURS: stopped = True; break
    ch = todo[i:i+JBATCH]
    for p, v in zip(ch, _score_chunk(ch)): cache[_key(*p)] = float(v)
    if b % 10 == 0:
        if not DRY_RUN: _save(cache)
        done = min(i + JBATCH, len(todo)); el = (time.time() - t0) / 60
        print(f"  {done}/{len(todo)} done | {done/max(el,1e-9):.0f} per minute | elapsed {el:.1f} min", flush=True)
        if AUTO_DOWNLOAD_EVERY_MIN and not DRY_RUN and (time.time() - last_dl) / 60 >= AUTO_DOWNLOAD_EVERY_MIN:
            try:
                from google.colab import files; files.download(PARTIAL); last_dl = time.time()
            except Exception as e: print("auto-download failed:", e)
if not DRY_RUN: _save(cache)
if stopped: raise SystemExit(f"Stopped cleanly after {MAX_RUN_HOURS} h; progress saved in {PARTIAL}. Start a new session, set RESUME_FROM_UPLOAD = True and run again.")
print("all judgements done:", len(cache))

In [ ]:
J = sf[["claim_id", "doc_id", "cond"]].copy()
for k in QUESTIONS: J["J_" + k] = [cache[_key(*p)] for p in PROMPTS[k]]
J["J_rel_minus_con"] = J.J_rel - J.J_con; J["J_sup_minus_con"] = J.J_sup - J.J_con
D = J.merge(S6D[["claim_id", "doc_id", "cond", "S", "C", "ERS"]], on=["claim_id", "doc_id", "cond"], how="inner"); print("merged items:", len(D))
D.to_csv(f"{OUT}/s6g_scifact_item_scores_nowindowtext.csv", index=False)
ALL = ["CONTRADICT", "IRRELEVANT", "NEI_ONTOPIC"]
def auc(negs, col, sign=1):
    x = D[D.cond.isin(["SUPPORT"] + negs)]; return roc_auc_score((x.cond == "SUPPORT").astype(int), sign * x[col])
CHECK = [("ERS, support vs contradict", auc(["CONTRADICT"], "ERS"), 0.828), ("reliability prompt, support vs contradict", auc(["CONTRADICT"], "J_trust"), 0.877),
         ("reliability prompt, pooled", auc(ALL, "J_trust"), 0.945), ("relevance prompt, pooled", auc(ALL, "J_rel"), 0.941)]
print("REPRODUCTION CHECK against the Step 6a table:")
for n, got, exp in CHECK: print(f"  {n}: recomputed {got:.3f} | Step 6a {exp:.3f} | difference {got-exp:+.3f}")
if any(abs(g - e) > 0.01 for _, g, e in CHECK): print("!!! WARNING: at least one value differs from Step 6a by more than 0.01. Do not use these results before the author has interpreted this.")
else: print("Reproduction check passed (all within 0.01).")

## Paired cluster bootstrap over claims: ERS minus each judge score

In [ ]:
SC = {"Similarity only (S)": ("S", 1), "Contradiction only (-C)": ("C", -1), "Judge: relevance prompt": ("J_rel", 1), "Judge: reliability prompt": ("J_trust", 1),
      "Judge: -contradiction prompt": ("J_con", -1), "Judge: relevance minus contradiction": ("J_rel_minus_con", 1), "Judge: support minus contradiction": ("J_sup_minus_con", 1)}
TASKS = {"SUPPORT vs CONTRADICT": ["CONTRADICT"], "SUPPORT vs IRRELEVANT": ["IRRELEVANT"], "SUPPORT vs NEI_ONTOPIC": ["NEI_ONTOPIC"], "SUPPORT vs all others (pooled)": ALL}
B = 2000; rng = np.random.default_rng(2026); res = []
for tn, negs in TASKS.items():
    x = D[D.cond.isin(["SUPPORT"] + negs)].reset_index(drop=True); y = (x.cond == "SUPPORT").astype(int).values
    cl, g = np.unique(x.claim_id.values, return_inverse=True); W = rng.multinomial(len(cl), np.ones(len(cl)) / len(cl), size=B)
    ers = x.ERS.values; a0 = roc_auc_score(y, ers)
    bs_e = np.array([roc_auc_score(y[W[b][g] > 0], ers[W[b][g] > 0], sample_weight=W[b][g][W[b][g] > 0]) for b in range(B)])
    for nm, (col, sg) in SC.items():
        s = sg * x[col].values; a1 = roc_auc_score(y, s)
        bs = np.array([roc_auc_score(y[W[b][g] > 0], s[W[b][g] > 0], sample_weight=W[b][g][W[b][g] > 0]) for b in range(B)])
        d = bs_e - bs; res.append(dict(task=tn, comparator=nm, AUROC_ERS=a0, AUROC_comparator=a1, diff_ERS_minus_comparator=a0 - a1, lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5)))
R = pd.DataFrame(res); print(R.round(3).to_string(index=False)); R.to_csv(f"{OUT}/s6g_scifact_paired.csv", index=False)
shutil.copy(PARTIAL, OUT) if os.path.exists(PARTIAL) else None
shutil.make_archive("rav_med_step6g_outputs", "zip", OUT)
try:
    from google.colab import files; files.download("rav_med_step6g_outputs.zip")
except ImportError: pass
print("Saved rav_med_step6g_outputs.zip")